### **Autor:** David Roca Tauste

---
---
# **ACTIVIDAD 5: NUEVO DETECTOR DE SPAM CON DATOS + REALISTAS**
---
---

## DATASOURCE

## ENTREGA 16: Escoge de tu cuenta de correo entre 3 y 5 mails que sean spam y completa hasta un total de 10 mails que no lo sean y los guardas en la carpeta datos en formato .eml. Les cambias el nombre por <3PrimerasLetrasDeTuNombre><3PrimerasLetrasDeTusApellidos>.eml. Tras eliminar información sensible los subes al recipiente compartido por todos (carpeta de aules de la práctica/datos). Además, edita el fichero compartido SPAMTrain.label y añade entradas donde los etiquetas (0 para spam y 1 para ham). Una vez que todos hayamos realizado esta aportación de nuevos datos, descargas el fichero de etiquetas y los ficheros con los mail y guardas los mail en la carpeta ./datos/training y el fichero SPAMTrain.label en ./datos. A modo de ejemplo estos son mis emails y sus etiquetas:
Ya he añadido los correos. Al final solo he añadido correos de spam por tema de privacidad.

## LA CLASE Email

In [6]:
import email
from bs4 import BeautifulSoup

class Email(object):
    CLRF = "\r\n\r\n"

    def __init__(self, archivo, categoria=None):
        self.categoria = categoria
        self.mail = email.message_from_binary_file(archivo)

    def subject(self):
        return self.mail.get("Subject")

    def body(self):
        payload = self.mail.get_payload()
        if self.mail.is_multipart():
            partes = [self._body_unico(parte) for parte in list(payload)]
        else:
            partes = [self._body_unico(self.mail)]
        partes_decodificadas = []
        for parte in partes:
            if len(parte) == 0:
                continue
            if isinstance(parte, bytes):
                partes_decodificadas.append(parte.decode("utf-8", errors="ignore"))
            else:
                partes_decodificadas.append(parte)
        return self.CLRF.join(partes_decodificadas)

    @staticmethod
    def _body_unico(parte):
        tipo_de_contenido = parte.get_content_type()
        try:
            body = parte.get_payload(decode=True)
        except Exception:
            body = ""
        if tipo_de_contenido == "text/html":
            return BeautifulSoup(body, "html.parser").text
        elif tipo_de_contenido == "text/plain":
            return body
        return ""

---
## ENTREGA 17: Crea el fichero objeto_email.py con el código anterior. Debes usarlo en el fichero genera_dataset.py que se encargará de procesar los emails guardados en los ficheros .eml que habrás preparado en la carpeta ./datos/training/ Para cada uno, el programa genera_dataset.py creará un ejemplo con dos campos label + texto. En el campo de texto aparecerán las palabras del subject del email unidas a las palabras de cuerpo. El campo label tendrá una indicación de si es spam o ham. La marca concreta que utilices puede ser la que tu quieras (un código numérico, una palabra, etc.). El algoritmo del programa genera_dataset.py será el siguiente:
- PASO 1. Para cada línea del fichero SPAMTrain.label:
- PASO 2. label, f ← primer y segundo campo de la línea
- PASO 3. mail ← objeto Email creado a partir del fichero ./datos/training/f
- PASO 4. Añade al fichero ./datos/dataset.csv una línea con: label + frases de subject y body

Entrega el código de ambos programas Python y el fichero dataset.csv generado al ejecutarlos.

In [7]:
import os
import csv

# Rutas
LABELS_FILE = "./SPAMTrain.label"
EMAILS_DIR = "./datos/training/"
OUTPUT_FILE = "./datos/dataset.csv"

def cargar_labels(ruta_labels):
    with open(ruta_labels, "r", encoding="utf-8") as f:
        lineas = f.readlines()
    return [linea.strip().split() for linea in lineas if linea.strip()]

def procesar_emails():
    ejemplos = []
    labels = cargar_labels(LABELS_FILE)

    for label, filename in labels:
        ruta_email = os.path.join(EMAILS_DIR, filename)
        try:
            with open(ruta_email, "rb") as archivo:
                mail = Email(archivo, categoria=label)
                subject = mail.subject() or ""
                body = mail.body() or ""
                texto = subject.strip() + " " + body.strip()
                ejemplos.append((label, texto.replace("\r", "").replace("\n", " ")))
        except Exception as e:
            print(f"Error procesando {filename}: {e}")

    return ejemplos

def guardar_dataset(ejemplos, ruta_salida):
    os.makedirs(os.path.dirname(ruta_salida), exist_ok=True)
    with open(ruta_salida, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["label", "texto"])
        for ejemplo in ejemplos:
            writer.writerow(ejemplo)

In [8]:
"""
ejemplos = procesar_emails()
guardar_dataset(ejemplos, OUTPUT_FILE)
print(f"Dataset generado con {len(ejemplos)} ejemplos.")
"""

'\nejemplos = procesar_emails()\nguardar_dataset(ejemplos, OUTPUT_FILE)\nprint(f"Dataset generado con {len(ejemplos)} ejemplos.")\n'

El código se encuentra arriba y el fichero 'dataset.csv' en la carpeta datos.

---
## ENTREGA 18. Usa el dataset para crear varios clasificadores de emails (al menos 6 distintos) en spam o ham. Tienes libertad para hacerlos de cualquiera de las formas que hemos visto en la unidad o en ejercicios anteriores de esta práctica (lógicamente, se puntúa que realices procesamientos correctos, etc.). Ten en cuenta además que uses más o menos librerías, no estaría mal que ahora que los datos van a ser más abundantes, descartes “palabras” muy comunes que no aportan nada y para ello piensa que nuestros spam usarán castellano o valenciano y los iniciales utilizan el inglés (por si descartas palabras, etc.) Si usas stop-words adicionales o personalizados, pásalos. Entrega el código.

In [9]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\davil\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [10]:
import pandas as pd
import string
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

# Cargar dataset
DATASET_PATH = "./datos/dataset.csv"
df = pd.read_csv(DATASET_PATH)

# Limpieza
df = df.dropna(subset=["label", "texto"])
df["texto"] = df["texto"].astype(str).str.lower()

# Stopwords
stop_words = (
    set(stopwords.words("english"))
    | set(stopwords.words("spanish"))
    | set(
        [
            "que",
            "de",
            "el",
            "en",
            "la",
            "los",
            "les",
            "per",
            "una",
            "un",
            "i",
            "es",
            "no",
            "sí",
            "com",
            "amb",
            "a",
            "al",
            "del",
            "dels",
            "o",
            "u",
        ]
    )
)


# Preprocesamiento adicional
def limpiar_texto(texto):
    texto = texto.translate(str.maketrans("", "", string.punctuation))
    palabras = texto.split()
    palabras = [p for p in palabras if p not in stop_words]
    return " ".join(palabras)


df["texto_limpio"] = df["texto"].apply(limpiar_texto)

# Separar datos
X = df["texto_limpio"]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Vectorización
vectorizer = TfidfVectorizer()
X_train_vect = vectorizer.fit_transform(X_train)
X_test_vect = vectorizer.transform(X_test)

# Lista de clasificadores
modelos = {
    "Naive Bayes": MultinomialNB(),
    "Regresión Logística": LogisticRegression(max_iter=1000),
    "SVM Lineal": LinearSVC(),
    "Árbol de Decisión": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "Gradient Boosting": GradientBoostingClassifier(),
}

# Entrenamiento
for nombre, modelo in modelos.items():
    print(f"\nEntrenando: {nombre}")
    modelo.fit(X_train_vect, y_train)


Entrenando: Naive Bayes

Entrenando: Regresión Logística

Entrenando: SVM Lineal

Entrenando: Árbol de Decisión

Entrenando: Random Forest

Entrenando: Gradient Boosting


---
## ENTREGA 19. Mide métricas de eficiencia de los clasificadores que hayas implementado.

In [11]:
# Evaluación
for nombre, modelo in modelos.items():
    pred = modelo.predict(X_test_vect)
    print(f"Accuracy: {accuracy_score(y_test, pred):.4f}")
    print(classification_report(y_test, pred))

Accuracy: 0.9030
              precision    recall  f1-score   support

           0       1.00      0.71      0.83       291
           1       0.87      1.00      0.93       575

    accuracy                           0.90       866
   macro avg       0.93      0.86      0.88       866
weighted avg       0.91      0.90      0.90       866

Accuracy: 0.9527
              precision    recall  f1-score   support

           0       0.99      0.87      0.93       291
           1       0.94      0.99      0.97       575

    accuracy                           0.95       866
   macro avg       0.96      0.93      0.95       866
weighted avg       0.95      0.95      0.95       866

Accuracy: 0.9850
              precision    recall  f1-score   support

           0       1.00      0.96      0.98       291
           1       0.98      1.00      0.99       575

    accuracy                           0.98       866
   macro avg       0.99      0.98      0.98       866
weighted avg       0.99

---
## ENTREGA 20. El mejor modelo, lo guardas en un fichero para usarlo en posibles aplicaciones. Investiga como se almacena un modelo en disco para usarlo posteriormente.

In [14]:
import joblib

mejor_modelo = None
mejor_score = 0
mejor_nombre = ""

for nombre, modelo in modelos.items():
    score = modelo.score(X_test_vect, y_test)
    if score > mejor_score:
        mejor_score = score
        mejor_modelo = modelo
        mejor_nombre = nombre

print(f"Mejor modelo: {mejor_nombre} con accuracy: {mejor_score:.4f}")

# Guardar modelo
joblib.dump(mejor_modelo, "mejor_modelo.pkl")
print(f"\nEl modelo '{mejor_nombre}' se ha guardado correctamente.")

Mejor modelo: SVM Lineal con accuracy: 0.9850

El modelo 'SVM Lineal' se ha guardado correctamente.
